# asinh Compression for Delay-Tap Ducking

asinh(x) = ln(x + sqrt(x² + 1))

asinh(-x) = -asinh(x)
asinh(x) ≈ x                  (x → 0)
asinh(x) ≈ sign(x)·ln(2|x|)   (|x| → ∞)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def asinh(x):
    return np.log(x + np.sqrt(x * x + 1.0))

_x = np.linspace(-6, 6, 4001)
assert np.allclose(asinh(_x), np.arcsinh(_x))

## 1. The asinh curve and its slope

d/dx asinh(x) = 1 / sqrt(x² + 1)

In [ ]:
x = np.linspace(-5, 5, 4001)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(x, asinh(x), label='asinh(x)')
ax[0].plot(x, x, color='k', linestyle='--', linewidth=1, label='y = x')
ax[0].set(title='asinh transfer', xlabel='x', ylabel='y')
ax[0].legend(); ax[0].grid(alpha=0.3)

slope = 1.0 / np.sqrt(x * x + 1.0)
ax[1].plot(x, slope, label='1 / sqrt(x² + 1)')
ax[1].set(title='slope (instantaneous gain)', xlabel='x', ylabel='dy/dx')
ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 2. Compression transfer curve

c_k(x) = asinh(k·x) / k
c_k(x) ≈ x                   (x → 0)
c_k(x) ≈ ln(2·k·x) / k       (x → ∞)

In [ ]:
def comp_curve(x, k):
    return asinh(k * x) / k

x = np.linspace(0, 1, 2001)
fig, ax = plt.subplots(figsize=(7, 5))
for k in (1, 2, 4, 8, 16, 32):
    ax.plot(x, comp_curve(x, k), label='k = %g' % k)
ax.plot(x, x, color='k', linestyle='--', linewidth=1, label='identity')
ax.set(title='asinh compression curve  c_k(x) = asinh(k·x) / k',
       xlabel='input level (normalized)', ylabel='output level (normalized)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 3. Duck gain and gain reduction

g(x) = asinh(k·x) / (k·x)
g(0) = 1
g(x) → 0   as x → ∞
GR(x) = 20·log10(g(x))
g_depth(x) = depth + (1 - depth)·g(x)

In [ ]:
def duck_gain(x, k, depth=0.0, eps=1e-12):
    xe = np.maximum(x, eps)
    g = asinh(k * xe) / (k * xe)
    return depth + (1.0 - depth) * g

x = np.linspace(1e-3, 1, 2001)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for k in (2, 4, 8, 16, 32):
    ax[0].plot(x, duck_gain(x, k), label='k = %g' % k)
    ax[1].plot(x, 20 * np.log10(duck_gain(x, k)), label='k = %g' % k)
ax[0].axhline(1.0, color='k', linewidth=1)
ax[0].set(title='duck gain  g(x) = asinh(k·x)/(k·x)',
          xlabel='dry level (normalized)', ylabel='tap gain (linear)')
ax[1].axhline(0.0, color='k', linewidth=1)
ax[1].set(title='gain reduction  20·log10(g)',
          xlabel='dry level (normalized)', ylabel='tap gain (dB)')
for a in ax:
    a.legend(); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 4. Dry sidechain envelope

In [ ]:
fs = 8000
T = 2.0
t = np.arange(0, T, 1.0 / fs)

def pluck_env(onset, decay, amp=1.0):
    return amp * np.exp(-(t - onset) / decay) * (t >= onset)

e1 = pluck_env(0.05, 0.15, 1.0)
e2 = pluck_env(0.70, 0.10, 0.7)
e3 = pluck_env(1.30, 0.20, 0.9)
dry_sig = (e1 * np.sin(2 * np.pi * 110.0 * t)
         + e2 * np.sin(2 * np.pi * 165.0 * t)
         + e3 * np.sin(2 * np.pi * 220.0 * t))

def one_pole_follow(x, atk_s, rel_s, fs):
    out = np.empty_like(x)
    y = 0.0
    ga = np.exp(-1.0 / (atk_s * fs))
    gr = np.exp(-1.0 / (rel_s * fs))
    for i, xi in enumerate(x):
        a = ga if xi > y else gr
        y = a * y + (1.0 - a) * xi
        out[i] = y
    return out

rect = np.abs(dry_sig)
rect = rect / max(rect.max(), 1e-9)
env = one_pole_follow(rect, 0.005, 0.080, fs)

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(t, dry_sig, color='0.7', linewidth=0.5, label='dry signal')
ax.plot(t, env, color='C0', label='sidechain envelope e(t)')
ax.set(title='Dry signal and sidechain envelope', xlabel='time (s)', ylabel='level')
ax.legend(loc='upper right'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 5. Duck gain over time

In [ ]:
k = 8.0
depth = 0.0

gain_inst = duck_gain(env, k, depth)
gain = one_pole_follow(gain_inst, 0.005, 0.080, fs)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t, env, color='C0', alpha=0.6, label='dry envelope e(t)')
ax.plot(t, gain, color='C3', label='tap gain g(t)  (k=%g)' % k)
ax.set(title='Delay-tap duck gain from dry sidechain',
       xlabel='time (s)', ylabel='level / gain')
ax.legend(loc='upper right'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 6. Applied to delay taps

In [ ]:
def multi_tap_delay(sig, tap_times_s, tap_gain, fb, fs):
    n = len(sig)
    taps = [int(d * fs) for d in tap_times_s]
    buf = sig.copy()
    wet = np.zeros(n)
    for i in range(n):
        v = 0.0
        for d in taps:
            if i - d >= 0:
                v += buf[i - d] * tap_gain
        wet[i] = v
        buf[i] = sig[i] + fb * v
    return wet

wet = multi_tap_delay(dry_sig, [0.18, 0.27], 0.6, 0.4, fs)
ducked_wet = wet * gain

fig, ax = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
ax[0].plot(t, dry_sig, color='C0')
ax[0].set(title='dry signal', ylabel='amp')
ax[1].plot(t, wet, color='C2')
ax[1].set(title='wet return (no ducking)', ylabel='amp')
ax[2].plot(t, ducked_wet, color='C3')
ax[2].set(title='wet return (asinh-ducked)', xlabel='time (s)', ylabel='amp')
for a in ax:
    a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 7. Comparison: asinh duck vs hard-threshold duck

In [ ]:
def hard_duck_gain(x, thresh, ratio, depth=0.0):
    y = np.where(x <= thresh, x, thresh + (x - thresh) / ratio)
    g = y / np.maximum(x, 1e-12)
    return depth + (1.0 - depth) * np.clip(g, 0.0, 1.0)

x = np.linspace(1e-3, 1, 2001)
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(x, 20 * np.log10(duck_gain(x, 8.0)), color='C3', label='asinh  k=8')
ax.plot(x, 20 * np.log10(hard_duck_gain(x, thresh=0.3, ratio=4.0)), color='C4',
        linestyle='--', label='hard knee  thr=0.30, ratio=4')
ax.axhline(0.0, color='k', linewidth=1)
ax.set(title='Duck gain reduction: smooth vs hard-knee',
       xlabel='dry level (normalized)', ylabel='tap gain (dB)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 8. Takeaways for the DSP core

g(x) = asinh(k·x) / (k·x)
g_depth(x) = depth + (1 - depth)·g(x)